In [1]:
import torch
import pandas as pd
from epiweeks import Week
import preprocess_data as prep
import model as mc
import matplotlib.pyplot as plt 
from loss_func import WeightedWISLossFromDistribution, LogNormalCRPSLoss

pd.options.mode.chained_assignment = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

regioes_estados = {
        'Sul': ['SC', 'PR', 'RS'],
        'Sudeste': ['SP', 'MG', 'RJ', 'ES'],
        'Nordeste': ['BA', 'CE', 'PE', 'PB', 'PI', 'RN', 'MA', 'AL', 'SE'],
        'Centro-Oeste': ['DF', 'MT', 'MS', 'GO'],
        'Norte': ['RO', 'AC', 'AM', 'RR', 'PA', 'AP', 'TO']
    } 
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

columns_to_normalize = ['casos','epiweek', 'biome', 'enso']

predict_n = 30
max_epiweek = 22
    
boxcox = False

TEST_YEAR = 2023


In [2]:
device

device(type='cuda')

In [ ]:
batch_size = 2
epochs = 200
media =True
verbose = 1
doenca = 'dengue'
min_delta = 0.002
patience= 25
lr = 2e-4
min_year = 2015
model_name = f'enso_22_V3'

In [4]:
df = prep.load_cases_data(filename= f'./data/{doenca}.csv.gz')

enso = prep.load_enso_weekly(filename='data/enso_weekly_forecast_up_25_07.csv')

enso_neutro = prep.load_enso_weekly(filename='data/enso_weekly_neutro.csv')

In [5]:
for region in regioes_estados.keys(): 

    print(region)

    label = f'{region}_{TEST_YEAR-1}_{model_name}'

    df_reg = df.loc[df.uf.isin(regioes_estados[region])]
    df_reg = df_reg.loc[df_reg.index >= pd.to_datetime(Week(2015,1).startdate())]


    X_train, y_train, X_future, norm_values, norm_enso =  prep.generate_regional_train_samples(df_reg, enso, 
                                                                                    TEST_YEAR,
                                                                                    max_epiweek = max_epiweek,
                                                                                    columns_to_normalize = columns_to_normalize,
                                                                                    min_year = min_year, 
                                                                                    boxcox = boxcox,
                                                                                    media = media)

    model = mc. LSTMWithFutureCovariatesV3(

                    hidden=64,

                    past_features=5,

                    future_cov_size=predict_n,

                    predict_n=predict_n,

                    dropout=0.2
                        )
    
    #model_path = f'./saved_models/trained_dengue_{region}_{TEST_YEAR-1}_enso_media.pt'
    #model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    #model.to(device)  
    
    trained_model, hist = mc.train_model(

            model=model,

            X_train=X_train,

            X_future=X_future,

            Y_train=y_train,

            label=label,

            batch_size=batch_size,

            epochs=epochs,

            lr=lr,

            patience=patience,

            verbose=verbose,

            doenca=doenca,

            #criterion =   LogNormalCRPSLoss().to(device)
)


Sul
Epoch 1/200 | Train: 0.6031 | Val: 0.4857
Epoch 2/200 | Train: 0.3151 | Val: 0.1788
Epoch 3/200 | Train: 0.1509 | Val: 0.1173
Epoch 4/200 | Train: 0.1060 | Val: 0.0942
Epoch 5/200 | Train: 0.0863 | Val: 0.0825
Epoch 6/200 | Train: 0.0761 | Val: 0.0745
Epoch 7/200 | Train: 0.0659 | Val: 0.0702
Epoch 8/200 | Train: 0.0671 | Val: 0.0663
Epoch 9/200 | Train: 0.0554 | Val: 0.0650
Epoch 10/200 | Train: 0.0544 | Val: 0.0632
Epoch 11/200 | Train: 0.0497 | Val: 0.0614
Epoch 12/200 | Train: 0.0481 | Val: 0.0609
Epoch 13/200 | Train: 0.0481 | Val: 0.0609
Epoch 14/200 | Train: 0.0517 | Val: 0.0593
Epoch 15/200 | Train: 0.0464 | Val: 0.0591
Epoch 16/200 | Train: 0.0440 | Val: 0.0584
Epoch 17/200 | Train: 0.0430 | Val: 0.0570
Epoch 18/200 | Train: 0.0446 | Val: 0.0559
Epoch 19/200 | Train: 0.0436 | Val: 0.0551
Epoch 20/200 | Train: 0.0431 | Val: 0.0549
Epoch 21/200 | Train: 0.0439 | Val: 0.0552
Epoch 22/200 | Train: 0.0426 | Val: 0.0542
Epoch 23/200 | Train: 0.0426 | Val: 0.0533
Epoch 24/200 | T